### Setup

In [ ]:
!hf auth login

In [ ]:
import wandb
wandb.login()

### GR-1 Pickup Grasp

In [ ]:
from google.colab import userdata
import os

# 1. Install Rerun and LeRobot Beforehand (due to version conflicts)
!pip install -q mujoco mink "qpsolvers[osqp]"
!pip install -q "rerun-sdk[all]==0.28.2" "imageio[ffmpeg]"
!pip install -q "transformers<5.0.0" "lerobot==0.4.3"
!pip install -q hydra-core lightning omegaconf h5py hdf5plugin
!pip install -q stable-pretraining stable-worldmodel torchcodec

# 2. Authenticate and Clone Private Repo
token = userdata.get("GITHUB_TOKEN")
github_url = f"https://{token}@github.com/vedpatwardhan/le-probe.git"
!git clone --recursive $github_url

print("✅ Environment Initialized.")
exit()

In [ ]:
%cd /content/le-probe/lewm
%env PYTHONPATH=/env/python:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup_grasp
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup_grasp
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup_grasp

In [ ]:
!python train_lewm.py \
    data=gr1 \
    data.dataset.repo_id="vedpatwardhan/gr1_pickup_grasp" \
    ++subdir=gr1_grasp_v3 \
    trainer.max_epochs=100 \
    loader.batch_size=128 \
    num_workers=10 \
    wandb.config.entity="vedpat3-none" \
    wandb.config.project="lewm-grasp-baseline" \
    wandb.config.log_model=False \
    ++rabc_kappa=0.01

In [ ]:
%%writefile consolidate_output.sh

mkdir -p lewm_grasp_baseline
cp -r outputs lewm_grasp_baseline
cp -r wandb lewm_grasp_baseline
cp environment.json lewm_grasp_baseline
cp manifold_diagnostics.csv lewm_grasp_baseline
cp requirements_frozen.txt lewm_grasp_baseline

In [ ]:
!bash consolidate_output.sh

In [ ]:
!cp -r lewm_grasp_baseline /content/drive/.../lewm_grasp_baseline

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

### GR-1 Reward Pred

In [ ]:
from google.colab import userdata
import os

# 1. Install Rerun and LeRobot Beforehand (due to version conflicts)
!pip install -q mujoco mink "qpsolvers[osqp]"
!pip install -q "rerun-sdk[all]==0.28.2" "imageio[ffmpeg]"
!pip install -q "transformers<5.0.0" "lerobot==0.4.3"
!pip install -q hydra-core lightning omegaconf h5py hdf5plugin
!pip install -q stable-pretraining stable-worldmodel torchcodec

# 2. Authenticate and Clone Private Repo
token = userdata.get("GITHUB_TOKEN")
github_url = f"https://{token}@github.com/vedpatwardhan/le-probe.git"
!git clone --recursive $github_url

print("✅ Environment Initialized.")
exit()

In [ ]:
%cd /content/le-probe/lewm
%env PYTHONPATH=/env/python:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [2]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_reward_pred
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_reward_pred
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_reward_pred

In [ ]:
!cp /content/drive/.../gr1-epoch=99-step=004400.ckpt .

In [ ]:
!python tune_reward_head.py \
    --ckpt gr1-epoch=99-step=004400.ckpt \
    --snapshots vedpatwardhan/gr1_reward_pred \
    --batch_size 32

In [ ]:
!cp gr1_reward_tuned.ckpt /content/drive/.../lewm_grasp_baseline

In [14]:
from google.colab import drive
drive.flush_and_unmount()

### Harvest Goals

In [ ]:
from google.colab import userdata
import os

# 1. Install Rerun and LeRobot Beforehand (due to version conflicts)
!pip install -q mujoco mink "qpsolvers[osqp]"
!pip install -q "rerun-sdk[all]==0.28.2" "imageio[ffmpeg]"
!pip install -q "transformers<5.0.0" "lerobot==0.4.3"
!pip install -q hydra-core lightning omegaconf h5py hdf5plugin
!pip install -q stable-pretraining stable-worldmodel torchcodec

# 2. Authenticate and Clone Private Repo
token = userdata.get("GITHUB_TOKEN")
github_url = f"https://{token}@github.com/vedpatwardhan/le-probe.git"
!git clone --recursive $github_url

print("✅ Environment Initialized.")
exit()

In [ ]:
!hf auth login

In [ ]:
%cd /content/le-probe/lewm
%env PYTHONPATH=/env/python:/content/le-probe/lewm:/content/le-probe/lewm/le_wm

In [ ]:
!cp /content/drive/.../gr1_reward_tuned_v2.ckpt .

In [ ]:
!python harvest_goals.py --model gr1_reward_tuned_v2.ckpt

In [ ]:
!cp goal_gallery.pth /content/drive/.../lewm_grasp_baseline

In [ ]:
!python diagnose_mpc.py --model gr1_reward_tuned_v2.ckpt --gallery goal_gallery.pth